# 05.2 XGBoost Root-Cause Sensitivity

Notebook 05 explains the selected `random_forest` model. This companion notebook asks a narrower robustness question: because XGBoost was close to random forest in cross-validated PR-AUC, do the leading sensor rankings hold if explanations are generated from the near-tie XGBoost model instead?

The selected production artifact remains `models/selected_model.joblib`; this notebook is a robustness check on the sensor ranking, not a second model-selection step.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

sys.path.insert(0, str(ROOT / "src"))

from yield_risk.config import load_config
from yield_risk.explainability import (
    compute_shap_values,
    global_feature_importance,
)
from yield_risk.root_cause import (
    fail_shap_lift,
    rank_root_cause_candidates,
    spc_flag_rate,
)

pd.set_option("display.max_colwidth", 120)

In [2]:
cfg = load_config(ROOT / "configs" / "config.yaml")
reports_dir = cfg.paths.reports_dir
models_dir = cfg.paths.models_dir

train = pd.read_csv(cfg.paths.splits_dir / "train.csv")
test = pd.read_csv(cfg.paths.splits_dir / "test.csv")
sensor_cols = [c for c in test.columns if c.startswith("sensor_")]

X_train = train[sensor_cols]
X_test = test[sensor_cols]
y_test = test["label"].to_numpy()

rf_candidates = pd.read_csv(reports_dir / "root_cause_candidates.csv")
rf_global = pd.read_csv(reports_dir / "shap_global_importance.csv")
xgb_pipeline = joblib.load(models_dir / "xgboost.joblib")

print(f"Loaded {len(X_train):,} train rows and {len(X_test):,} test rows.")
print(f"Loaded XGBoost model from {models_dir / 'xgboost.joblib'}")
print(f"Selected-model root-cause baseline: {reports_dir / 'root_cause_candidates.csv'}")

Loaded 1,331 train rows and 236 test rows.
Loaded XGBoost model from models\xgboost.joblib
Selected-model root-cause baseline: reports\root_cause_candidates.csv


## Method

The sensitivity path mirrors notebook 05 exactly, except the fitted model is `models/xgboost.joblib` rather than the selected random-forest artifact. The comparison uses the same processed test rows, the same SHAP utility, the same fail/pass SHAP lift calculation, and the same SPC flag-rate calculation.

In [3]:
xgb_explanations = compute_shap_values(xgb_pipeline, X_train, X_test)
xgb_global = global_feature_importance(xgb_explanations)
xgb_lift = fail_shap_lift(xgb_explanations, y_test)
flag_rates = spc_flag_rate(test, sensor_cols)
xgb_candidates = rank_root_cause_candidates(xgb_global, xgb_lift, flag_rates)

print(f"Computed XGBoost SHAP for {len(X_test):,} test rows and {len(sensor_cols):,} sensors.")
xgb_candidates.head(10)

Background dataset has 1331 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1331 when initializing the masker.


Computed XGBoost SHAP for 236 test rows and 590 sensors.


,sensor,mean_abs_shap,shap_lift,spc_flag_rate,composite_score
0,sensor_059,0.362427,0.670049,0.016949,0.861538
1,sensor_033,0.319325,0.119129,0.016949,0.555413
2,sensor_021,0.179033,0.058455,0.021186,0.350087
3,sensor_103,0.208244,0.080912,0.004237,0.338903
4,sensor_587,0.191798,-0.002579,0.016949,0.327295
5,sensor_213,0.158108,-0.050123,0.008475,0.271335
6,sensor_460,0.119021,0.058301,0.021186,0.267225
7,sensor_019,0.145034,0.106879,0.004237,0.263325
8,sensor_130,0.161904,0.081541,0.000000,0.259869
9,sensor_205,0.130892,0.112753,0.004237,0.246445


## Ranking Overlap

A sensor ranking is more robust when a near-tie model family surfaces many of the same leading sensors. The tables below compare the selected random forest from notebook 05 with the close XGBoost challenger from notebook 04.

In [4]:
def top_sensor_set(df: pd.DataFrame, n: int) -> set[str]:
    """Return the top-n sensor names from a root-cause candidate table."""
    return set(df.head(n)["sensor"].tolist())


summary_rows = []
for n in (5, 10, 20):
    rf_top = top_sensor_set(rf_candidates, n)
    xgb_top = top_sensor_set(xgb_candidates, n)
    overlap = sorted(rf_top & xgb_top)
    union = rf_top | xgb_top
    summary_rows.append(
        {
            "top_n": n,
            "overlap_count": len(overlap),
            "jaccard": len(overlap) / len(union),
            "overlap_sensors": ", ".join(overlap),
        }
    )

sensitivity_summary = pd.DataFrame(summary_rows)

rf_ranked = rf_candidates.reset_index().rename(
    columns={
        "index": "rf_rank_zero_based",
        "composite_score": "rf_composite_score",
        "mean_abs_shap": "rf_mean_abs_shap",
        "shap_lift": "rf_shap_lift",
    }
)
rf_ranked["rf_rank"] = rf_ranked["rf_rank_zero_based"] + 1
xgb_ranked = xgb_candidates.reset_index().rename(
    columns={
        "index": "xgb_rank_zero_based",
        "composite_score": "xgb_composite_score",
        "mean_abs_shap": "xgb_mean_abs_shap",
        "shap_lift": "xgb_shap_lift",
    }
)
xgb_ranked["xgb_rank"] = xgb_ranked["xgb_rank_zero_based"] + 1

detail_cols = [
    "sensor",
    "rf_rank",
    "xgb_rank",
    "rf_composite_score",
    "xgb_composite_score",
    "rf_mean_abs_shap",
    "xgb_mean_abs_shap",
    "rf_shap_lift",
    "xgb_shap_lift",
]
sensitivity_detail = (
    rf_ranked.merge(xgb_ranked, on="sensor", how="outer")
    .assign(best_rank=lambda df: df[["rf_rank", "xgb_rank"]].min(axis=1))
    .sort_values(["best_rank", "sensor"])
    [detail_cols]
)

display(sensitivity_summary)
display(sensitivity_detail.head(15))

,top_n,overlap_count,jaccard,overlap_sensors
0,5,4,0.666667,"sensor_021, sensor_033, sensor_059, sensor_103"
1,10,6,0.428571,"sensor_021, sensor_033, sensor_059, sensor_103, sensor_130, sensor_205"
2,20,11,0.379310,"sensor_021, sensor_031, sensor_033, sensor_059, sensor_103, sensor_130, sensor_205, sensor_213, sensor_431, sensor_5..."


,sensor,rf_rank,xgb_rank,rf_composite_score,xgb_composite_score,rf_mean_abs_shap,xgb_mean_abs_shap,rf_shap_lift,xgb_shap_lift
47,sensor_059,1,1,0.861538,0.861538,0.010310,0.362427,0.018617,0.670049
30,sensor_033,2,2,0.487369,0.555413,0.007462,0.319325,0.003970,0.119129
19,sensor_021,5,3,0.289893,0.350087,0.003928,0.179033,0.001394,0.058455
85,sensor_103,3,4,0.356394,0.338903,0.006186,0.208244,0.002544,0.080912
28,sensor_031,4,13,0.306826,0.223322,0.005764,0.147829,-0.001693,-0.043283
240,sensor_587,20,5,0.171040,0.327295,0.002148,0.191798,-0.000330,-0.002579
52,sensor_064,6,94,0.275226,0.091760,0.003235,0.017987,0.003524,0.012077
160,sensor_213,18,6,0.176242,0.271335,0.002864,0.158108,-0.000408,-0.050123
154,sensor_205,7,10,0.260749,0.246445,0.003759,0.130892,0.003915,0.112753
206,sensor_460,27,7,0.150987,0.267225,0.001348,0.119021,0.000540,0.058301


In [5]:
rf_top5 = rf_candidates.head(5)["sensor"].tolist()
xgb_top5 = xgb_candidates.head(5)["sensor"].tolist()
top10_overlap = int(sensitivity_summary.loc[sensitivity_summary["top_n"] == 10, "overlap_count"].iloc[0])

print("Selected RF top 5:", ", ".join(rf_top5))
print("XGBoost sensitivity top 5:", ", ".join(xgb_top5))
print(f"Top-10 overlap: {top10_overlap}/10")

if rf_top5[0] == xgb_top5[0]:
    print(f"Both model families rank {rf_top5[0]} as the leading candidate.")
else:
    print("The leading candidate changes across model families; treat root-cause rank order as unstable.")

Selected RF top 5: sensor_059, sensor_033, sensor_103, sensor_031, sensor_021
XGBoost sensitivity top 5: sensor_059, sensor_033, sensor_021, sensor_103, sensor_587
Top-10 overlap: 6/10
Both model families rank sensor_059 as the leading candidate.


## Interpretation

The XGBoost sensitivity check supports the selected-model sensor ranking.
Both tree families rank `sensor_059` as the leading candidate, and the top-five
lists overlap on four sensors: `sensor_059`, `sensor_033`, `sensor_103`, and
`sensor_021`. The leading sensors hold across both model families rather than
depending on the random forest alone.

`sensor_059` remains first and the top-k overlap stays strong (6 of 10 in the
top ten). SECOM sensor names are anonymous, so the next step is mapping the
leading sensor set to tool history, recipe context, chamber state, maintenance
logs, and lot genealogy before changing process settings.